# LMA Phase 2 - Kaggle training and evaluation

Attach the private dataset created from `lma_phase2_kaggle_upload.zip`, select a GPU accelerator, and run the cells in order. Checkpoints are written to `/kaggle/working/phase2_runs`.

In [ ]:
import subprocess
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
from pathlib import Path
import shutil

manifests = sorted(Path('/kaggle/input').rglob('lma_phase2_bundle_manifest.json'))
if manifests:
    BUNDLE_ROOT = manifests[0].parent
else:
    archives = sorted(Path('/kaggle/input').rglob('lma_phase2_kaggle_upload.zip'))
    if not archives:
        raise FileNotFoundError('Attach the uploaded LMA Phase 2 Kaggle dataset first')
    extraction_root = Path('/kaggle/working/lma_phase2_bundle')
    extraction_root.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(archives[0]), extraction_root)
    manifests = sorted(extraction_root.rglob('lma_phase2_bundle_manifest.json'))
    BUNDLE_ROOT = manifests[0].parent

# Ensure runner supports compressed, decompressed, or directory-extracted splits
runner_src = BUNDLE_ROOT / 'kaggle' / 'run_phase2_kaggle.py'
RUNNER = Path('/kaggle/working/run_phase2_kaggle.py')
runner_text = runner_src.read_text(encoding='utf-8')
if 'find_file_in_dir' not in runner_text:
    replacement_finder = '''def find_split_file(root: Path, language: str, split: str) -> Path:
    """Locate split file whether compressed (.jsonl.gz), decompressed (.jsonl), or extracted as a directory."""
    def find_file_in_dir(d: Path) -> Path | None:
        if not d.exists():
            return None
        if d.is_file():
            return d
        if d.is_dir():
            for direct in (d / f"{split}.jsonl", d / f"{split}.jsonl.gz", d / d.name):
                if direct.is_file():
                    return direct
            nested = [f for f in d.rglob("*") if f.is_file() and not f.name.startswith(".")]
            if nested:
                return nested[0]
        return None

    for name in (f"{split}.jsonl.gz", f"{split}.jsonl"):
        candidate = root / "data" / language / name
        f = find_file_in_dir(candidate)
        if f is not None:
            return f

    for candidate in root.rglob(f"*{split}*.jsonl*"):
        if candidate.is_file() and not candidate.name.startswith("."):
            return candidate

    for candidate in Path("/kaggle/input").rglob(f"*{split}*.jsonl*"):
        if candidate.is_file() and not candidate.name.startswith("."):
            return candidate

    raise FileNotFoundError(f"Could not find {split} split file for {language} in {root}")
'''
    import re
    if 'def find_split_file' in runner_text:
        runner_text = re.sub(r'def find_split_file\(.*?
(?=def validate_bundle)', replacement_finder + '

', runner_text, flags=re.DOTALL)
    else:
        runner_text = runner_text.replace('def validate_bundle(', replacement_finder + '

def validate_bundle(')
        runner_text = runner_text.replace(
            'root / "data" / language / "validation.jsonl.gz",',
            'find_split_file(root, language, "validation"),'
        ).replace(
            'root / "data" / language / "test.jsonl.gz",',
            'find_split_file(root, language, "test"),'
        ).replace(
            'str(root / "data" / language / "test.jsonl.gz"),',
            'str(find_split_file(root, language, "test")),'
        )
RUNNER.write_text(runner_text, encoding='utf-8')

print('Bundle:', BUNDLE_ROOT)
print('Runner:', RUNNER)

## Pack the Nepali corpus on Kaggle

This CPU preprocessing step creates resumable uint16 streams in `/kaggle/working`.

In [ ]:
import importlib.util, os
from pathlib import Path
if importlib.util.find_spec('sentencepiece') is None:
    subprocess.run(['python', '-m', 'pip', 'install', '-q', 'sentencepiece'], check=True)

def resolve_split(root: Path, language: str, split: str) -> Path:
    def find_file_in_dir(d: Path) -> Path | None:
        if not d.exists():
            return None
        if d.is_file():
            return d
        if d.is_dir():
            for direct in (d / f"{split}.jsonl", d / f"{split}.jsonl.gz", d / d.name):
                if direct.is_file():
                    return direct
            nested = [f for f in d.rglob("*") if f.is_file() and not f.name.startswith(".")]
            if nested:
                return nested[0]
        return None

    for name in (f"{split}.jsonl.gz", f"{split}.jsonl"):
        candidate = root / "data" / language / name
        f = find_file_in_dir(candidate)
        if f is not None:
            return f

    for candidate in root.rglob(f"*{split}*.jsonl*"):
        if candidate.is_file() and not candidate.name.startswith("."):
            return candidate

    for candidate in Path("/kaggle/input").rglob(f"*{split}*.jsonl*"):
        if candidate.is_file() and not candidate.name.startswith("."):
            return candidate

    raise FileNotFoundError(f"Cannot find {split} split file for {language} in {root}")

train_path = resolve_split(BUNDLE_ROOT, 'nepali', 'train')
val_path = resolve_split(BUNDLE_ROOT, 'nepali', 'validation')
test_path = resolve_split(BUNDLE_ROOT, 'nepali', 'test')
print(f'Using splits:
  train: {train_path}
  val: {val_path}
  test: {test_path}')

DATA_ROOT = Path('/kaggle/working/phase2_data')
packed = DATA_ROOT / 'nepali' / 'train.manifest.json'
if not packed.exists():
    env = os.environ.copy()
    env['PYTHONPATH'] = str(BUNDLE_ROOT / 'repo' / 'src')
    subprocess.run([
        'python', '-m', 'lma_phase2.prepare_data',
        '--input', f"train={train_path}",
        '--input', f"validation={val_path}",
        '--input', f"test={test_path}",
        '--tokenizer-model', str(BUNDLE_ROOT / 'repo/nepali/tokenizer/nepali.model'),
        '--output-dir', str(DATA_ROOT / 'nepali'), '--language', 'ne'
    ], check=True, cwd=BUNDLE_ROOT / 'repo', env=env)
print('Packed data:', DATA_ROOT)

## Train or resume

Train Nepali in one or more Kaggle sessions. If a previous notebook output is attached, set `RESUME_ROOT` to the directory containing its `phase2_runs` folder.

In [ ]:
LANGUAGE = 'nepali'  # 'hindi', 'nepali', or 'all'
RESUME_ROOT = None  # e.g. '/kaggle/input/previous-phase2-output'
command = [
    'python', str(RUNNER),
    '--bundle-root', str(BUNDLE_ROOT),
    '--language', LANGUAGE,
    '--data-root', str(DATA_ROOT),
    '--mode', 'train',
    '--device', 'cuda',
]
if RESUME_ROOT:
    command.extend(['--resume-root', RESUME_ROOT])
subprocess.run(command, check=True)

## Evaluate completed checkpoints

This computes held-out CE/PPL/BPB, generation metrics and samples, plus attention heatmaps and summaries. Kaggle internet must be enabled if the two small evaluation packages are missing.

In [ ]:
import importlib.util
missing = [name for name in ('sentencepiece', 'sacrebleu') if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run(['python', '-m', 'pip', 'install', '-q', *missing], check=True)
subprocess.run([
    'python', str(RUNNER),
    '--bundle-root', str(BUNDLE_ROOT),
    '--language', LANGUAGE,
    '--data-root', str(DATA_ROOT),
    '--mode', 'evaluate',
    '--device', 'cuda',
], check=True)

## Package outputs

Download this archive and/or save the notebook as a version so the checkpoints can be attached to a later session.

In [ ]:
import zipfile
output_zip = Path('/kaggle/working/lma_phase2_outputs.zip')
with zipfile.ZipFile(output_zip, 'w', compression=zipfile.ZIP_STORED) as archive:
    report_dir = Path('/kaggle/working/phase2_report')
    if report_dir.exists():
        for path in report_dir.rglob('*'):
            if path.is_file():
                archive.write(path, path.relative_to('/kaggle/working'))
    runs_dir = Path('/kaggle/working/phase2_runs')
    for run_dir in runs_dir.glob('*') if runs_dir.exists() else []:
        metadata_names = ('latest.json', 'training_log.csv', 'resolved_config.json', 'run_environment.json')
        for name in metadata_names:
            path = run_dir / name
            if path.exists():
                archive.write(path, path.relative_to('/kaggle/working'))
        latest_path = run_dir / 'latest.json'
        if latest_path.exists():
            latest = __import__('json').loads(latest_path.read_text())
            checkpoint = run_dir / latest['checkpoint']
            archive.write(checkpoint, checkpoint.relative_to('/kaggle/working'))
print(output_zip, output_zip.stat().st_size, 'bytes')